In [1]:
import pandas as pd

In [ ]:
repo_with_cache = pd.read_csv('repos_with_cache_data.csv')
repo_without_cache = pd.read_csv('repos_without_cache_data.csv')
repo_with_cache = repo_with_cache[['owner_repo']]
repo_without_cache = repo_without_cache[['owner_repo']]
repo_with_cache

,owner_repo
0,lrusnac/hn-notifier
1,iotexproject/iotex-desktop-wallet
2,dotintent/react-native-ble-plx
3,leo-editor/leo-editor
4,mruby/mruby
...,...
261,jenkinsci/dingtalk-plugin
262,fgnt/pb_bss
263,huggingface/transformers
264,wechaty/getting-started


In [ ]:
import os
import time
import logging
from datetime import datetime, timezone
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import json

import pandas as pd
from github import Github, GithubException, RateLimitExceededException
from git import Repo, GitCommandError
from tqdm import tqdm

# Configuration
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN', '#')
CUT_OFF_DATE = datetime(2025, 5, 16, tzinfo=timezone.utc)
CUT_OFF_ISO = CUT_OFF_DATE.isoformat()
REPO_BASE_DIR = Path("./cloned_repos")
REPO_BASE_DIR.mkdir(exist_ok=True, parents=True)

# Parallel processing settings
MAX_WORKERS = 8  # Adjust based on your rate limit
RATE_LIMIT_BUFFER = 100  # Keep this many requests in reserve

# Initialize GitHub client & logging
g = Github(GITHUB_TOKEN, per_page=100)
logging.basicConfig(
    filename='repo_metrics.log',
    level=logging.ERROR,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Cache for processed repos
CACHE_FILE = "repo_cache.json"

def load_cache():
    """Load processed repositories from cache"""
    if os.path.exists(CACHE_FILE):
        try:
            with open(CACHE_FILE, 'r') as f:
                return json.load(f)
        except:
            return {}
    return {}

def save_cache(cache):
    """Save processed repositories to cache"""
    with open(CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

def check_rate_limit():
    """Check if we have enough rate limit remaining"""
    limit = g.get_rate_limit().core
    if limit.remaining < RATE_LIMIT_BUFFER:
        reset = limit.reset
        wait = max(5, (reset - datetime.now(reset.tzinfo)).total_seconds() + 5)
        logging.warning(f"Rate limit low ({limit.remaining}); sleeping for {wait:.0f}s")
        time.sleep(wait)

def handle_rate_limit():
    """Handle rate limit exceeded"""
    limit = g.get_rate_limit().core
    reset = limit.reset
    wait = max(5, (reset - datetime.now(reset.tzinfo)).total_seconds() + 5)
    logging.warning(f"Rate limit hit; sleeping for {wait:.0f}s")
    time.sleep(wait)

def clone_repo_shallow(owner_repo):
    """Clone repo with minimal depth for faster cloning"""
    name = owner_repo.split('/')[-1]
    target = REPO_BASE_DIR / name
    if not target.exists():
        url = f"https://{GITHUB_TOKEN}@github.com/{owner_repo}.git"
        try:
            # Use single-branch and depth=1 for speed
            return Repo.clone_from(url, str(target), depth=1, single_branch=True)
        except GitCommandError as e:
            logging.error(f"Clone failed ({owner_repo}): {e}")
            return None
    return Repo(str(target))

def get_local_metrics_fast(repo):
    """Get commit metrics using git log instead of iterating"""
    try:
        # Use git log with date filtering for much faster results
        commits = list(repo.iter_commits('HEAD', until=CUT_OFF_ISO, max_count=10000))
        
        # Get unique authors more efficiently
        authors = set()
        for commit in commits:
            if commit.author and commit.author.email:
                authors.add(commit.author.email)
        
        return len(commits), len(authors)
    except Exception as e:
        logging.error(f"Local metrics error: {e}")
        return 0, 0

def safe_count_paginated_optimized(paginated_list, filter_fn=lambda x: True, max_retries=3):
    """Safely count paginated results with proper error handling"""
    for attempt in range(max_retries):
        try:
            total = 0
            for item in paginated_list:
                if filter_fn(item):
                    total += 1
            return total
        except RateLimitExceededException:
            if attempt < max_retries - 1:
                handle_rate_limit()
                continue
            else:
                logging.error("Max retries exceeded for rate limit")
                return 0
        except Exception as e:
            logging.error(f"Error in pagination (attempt {attempt + 1}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
                continue
            else:
                return 0
    return 0

def get_github_metrics_optimized(owner_repo):
    """Optimized GitHub metrics collection with proper pagination handling"""
    m = {
        "owner_repo": owner_repo,
        "repository": owner_repo.split('/')[-1],
        "snapshot_date": CUT_OFF_ISO,
        "language": None,
        "stars_at_cutoff": 0,
        "forks_at_cutoff": 0,
        "total_prs": 0,
        "closed_prs": 0,
        "total_issues": 0,
        "closed_issues": 0
    }
    
    try:
        repo = g.get_repo(owner_repo)
        m["language"] = repo.language
        
        # Get current stars/forks (much faster than iterating)
        current_stars = repo.stargazers_count
        current_forks = repo.forks_count
        
        # Estimate historical count (rough approximation)
        m["stars_at_cutoff"] = current_stars  # Approximation
        m["forks_at_cutoff"] = current_forks  # Approximation
        
        # Get PR counts with proper pagination handling
        try:
            # Try to use totalCount first (fastest method)
            all_prs = repo.get_pulls(state="all")
            closed_prs = repo.get_pulls(state="closed")
            
            # Some repositories might not have totalCount, so try it carefully
            if hasattr(all_prs, 'totalCount') and all_prs.totalCount is not None:
                m["total_prs"] = all_prs.totalCount
            else:
                # Fallback to manual counting with proper pagination
                m["total_prs"] = safe_count_paginated_optimized(all_prs)
            
            if hasattr(closed_prs, 'totalCount') and closed_prs.totalCount is not None:
                m["closed_prs"] = closed_prs.totalCount
            else:
                # Fallback to manual counting with proper pagination
                m["closed_prs"] = safe_count_paginated_optimized(closed_prs)
                
        except Exception as e:
            logging.error(f"Error getting PR counts for {owner_repo}: {e}")
            # Fallback to manual counting
            try:
                all_prs = repo.get_pulls(state="all")
                closed_prs = repo.get_pulls(state="closed")
                m["total_prs"] = safe_count_paginated_optimized(all_prs)
                m["closed_prs"] = safe_count_paginated_optimized(closed_prs)
            except:
                m["total_prs"] = 0
                m["closed_prs"] = 0
        
        # Get issue counts with proper pagination (excluding PRs)
        try:
            all_issues = repo.get_issues(state="all")
            closed_issues = repo.get_issues(state="closed")
            
            # Filter out PRs from issues (issues API includes PRs)
            def is_pure_issue(issue):
                return issue.pull_request is None
            
            m["total_issues"] = safe_count_paginated_optimized(all_issues, is_pure_issue)
            m["closed_issues"] = safe_count_paginated_optimized(closed_issues, is_pure_issue)
            
        except Exception as e:
            logging.error(f"Error getting issue counts for {owner_repo}: {e}")
            m["total_issues"] = 0
            m["closed_issues"] = 0
            
    except RateLimitExceededException:
        handle_rate_limit()
        return get_github_metrics_optimized(owner_repo)
    except GithubException as e:
        logging.error(f"GitHub API error ({owner_repo}): {e.data.get('message', e) if hasattr(e, 'data') else str(e)}")
    except Exception as e:
        logging.error(f"Unexpected error ({owner_repo}): {e}")
    
    return m

def process_single_repo(owner_repo, cache, use_exact_historical=False):
    """Process a single repository with option for exact historical data"""
    # Check cache first
    cache_key = f"{owner_repo}_{use_exact_historical}"
    if cache_key in cache:
        return cache[cache_key]
    
    check_rate_limit()
    
    metrics = get_github_metrics_comprehensive(owner_repo, use_exact_historical)
    
    # Get local metrics (commits/contributors)
    try:
        repo = clone_repo_shallow(owner_repo)
        if repo:
            commits, authors = get_local_metrics_fast(repo)
            metrics.update({
                "commits": commits,
                "contributors": authors
            })
        else:
            metrics.update({
                "commits": 0,
                "contributors": 0
            })
    except Exception as e:
        logging.error(f"Error processing local repo ({owner_repo}): {e}")
        metrics.update({
            "commits": 0,
            "contributors": 0
        })
    
    # Cache the result
    cache[cache_key] = metrics
    return metrics

def process_repositories_parallel(df, label, use_exact_historical=False):
    """Process repositories in parallel with comprehensive pagination handling"""
    cache = load_cache()
    results = []
    
    # Filter out already processed repos
    cache_key_suffix = f"_{use_exact_historical}"
    remaining_repos = [repo for repo in df["owner_repo"] 
                      if f"{repo}{cache_key_suffix}" not in cache]
    
    print(f"Processing {len(remaining_repos)} repositories ({len(df) - len(remaining_repos)} already cached)")
    print(f"Using exact historical data: {use_exact_historical}")
    
    if not remaining_repos:
        # All repos are cached
        results = [cache[f"{repo}{cache_key_suffix}"] for repo in df["owner_repo"]]
    else:
        # Process remaining repos in parallel
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # Submit all tasks
            future_to_repo = {
                executor.submit(process_single_repo, repo, cache, use_exact_historical): repo 
                for repo in remaining_repos
            }
            
            # Process completed tasks with progress bar
            with tqdm(total=len(remaining_repos), desc=f"Processing {label}", unit="repo") as pbar:
                for future in as_completed(future_to_repo):
                    repo = future_to_repo[future]
                    try:
                        result = future.result()
                        results.append(result)
                        
                        # Update progress bar
                        pbar.set_postfix({
                            "repo": repo.split('/')[-1],
                            "stars": result["stars_at_cutoff"],
                            "commits": result.get("commits", 0),
                            "total_prs": result["total_prs"],
                            "total_issues": result["total_issues"]
                        })
                        pbar.update(1)
                        
                        # Save cache periodically
                        if len(results) % 10 == 0:
                            save_cache(cache)
                            
                    except Exception as e:
                        logging.error(f"Error processing {repo}: {e}")
                        pbar.update(1)
        
        # Add cached results
        for repo in df["owner_repo"]:
            cache_key = f"{repo}{cache_key_suffix}"
            if cache_key in cache and repo not in [r["owner_repo"] for r in results]:
                results.append(cache[cache_key])
    
    # Save final cache
    save_cache(cache)
    
    # Save results
    df_results = pd.DataFrame(results)
    df_results.to_csv(f"metrics_{label}.csv", index=False)
    
    print(f"Completed {label}: {len(results)} repositories processed")
    return df_results

def get_exact_historical_stars_forks(repo, cutoff_date):
    """
    Get exact historical star/fork counts (SLOW - use only if needed)
    This is the original slow method for exact historical data
    """
    stars_count = 0
    forks_count = 0
    
    try:
        # Count stars before cutoff with proper pagination
        stars_count = safe_count_paginated_optimized(
            repo.get_stargazers_with_dates(),
            lambda star: star.starred_at <= cutoff_date
        )
        
        # Count forks before cutoff with proper pagination
        forks_count = safe_count_paginated_optimized(
            repo.get_forks(),
            lambda fork: fork.created_at <= cutoff_date
        )
                
    except Exception as e:
        logging.error(f"Error getting exact historical data: {e}")
    
    return stars_count, forks_count

def get_github_metrics_comprehensive(owner_repo, use_exact_historical=False):
    """
    Comprehensive GitHub metrics with option for exact historical data
    
    Args:
        owner_repo: Repository in format 'owner/repo'
        use_exact_historical: If True, gets exact historical star/fork counts (SLOW)
    """
    m = {
        "owner_repo": owner_repo,
        "repository": owner_repo.split('/')[-1],
        "snapshot_date": CUT_OFF_ISO,
        "language": None,
        "stars_at_cutoff": 0,
        "forks_at_cutoff": 0,
        "total_prs": 0,
        "closed_prs": 0,
        "total_issues": 0,
        "closed_issues": 0
    }
    
    try:
        repo = g.get_repo(owner_repo)
        m["language"] = repo.language
        
        # Get star/fork counts
        if use_exact_historical:
            # Exact historical data (slow but accurate)
            stars_count, forks_count = get_exact_historical_stars_forks(repo, CUT_OFF_DATE)
            m["stars_at_cutoff"] = stars_count
            m["forks_at_cutoff"] = forks_count
        else:
            # Current counts (fast approximation)
            m["stars_at_cutoff"] = repo.stargazers_count
            m["forks_at_cutoff"] = repo.forks_count
        
        # Get PR counts with comprehensive error handling
        try:
            all_prs = repo.get_pulls(state="all")
            closed_prs = repo.get_pulls(state="closed")
            
            # Try totalCount first (fastest)
            if hasattr(all_prs, 'totalCount') and all_prs.totalCount is not None:
                m["total_prs"] = all_prs.totalCount
            else:
                m["total_prs"] = safe_count_paginated_optimized(all_prs)
            
            if hasattr(closed_prs, 'totalCount') and closed_prs.totalCount is not None:
                m["closed_prs"] = closed_prs.totalCount
            else:
                m["closed_prs"] = safe_count_paginated_optimized(closed_prs)
                
        except Exception as e:
            logging.error(f"Error getting PR counts for {owner_repo}: {e}")
            try:
                # Second attempt with fresh API calls
                all_prs = repo.get_pulls(state="all")
                closed_prs = repo.get_pulls(state="closed")
                m["total_prs"] = safe_count_paginated_optimized(all_prs)
                m["closed_prs"] = safe_count_paginated_optimized(closed_prs)
            except Exception as e2:
                logging.error(f"Second attempt failed for PR counts {owner_repo}: {e2}")
                m["total_prs"] = 0
                m["closed_prs"] = 0
        
        # Get issue counts with comprehensive error handling (excluding PRs)
        try:
            all_issues = repo.get_issues(state="all")
            closed_issues = repo.get_issues(state="closed")
            
            # Filter function to exclude PRs
            def is_pure_issue(issue):
                return issue.pull_request is None
            
            m["total_issues"] = safe_count_paginated_optimized(all_issues, is_pure_issue)
            m["closed_issues"] = safe_count_paginated_optimized(closed_issues, is_pure_issue)
            
        except Exception as e:
            logging.error(f"Error getting issue counts for {owner_repo}: {e}")
            try:
                # Second attempt with fresh API calls
                all_issues = repo.get_issues(state="all")
                closed_issues = repo.get_issues(state="closed")
                m["total_issues"] = safe_count_paginated_optimized(all_issues, lambda i: i.pull_request is None)
                m["closed_issues"] = safe_count_paginated_optimized(closed_issues, lambda i: i.pull_request is None)
            except Exception as e2:
                logging.error(f"Second attempt failed for issue counts {owner_repo}: {e2}")
                m["total_issues"] = 0
                m["closed_issues"] = 0
            
    except RateLimitExceededException:
        handle_rate_limit()
        return get_github_metrics_comprehensive(owner_repo, use_exact_historical)
    except GithubException as e:
        logging.error(f"GitHub API error ({owner_repo}): {e.data.get('message', e) if hasattr(e, 'data') else str(e)}")
    except Exception as e:
        logging.error(f"Unexpected error ({owner_repo}): {e}")
    
    return m


In [ ]:
process_repositories_parallel(repo_with_cache, "with_cache", use_exact_historical=False)
process_repositories_parallel(repo_without_cache, "without_cache", use_exact_historical=False)

Processing 266 repositories (0 already cached)
Using exact historical data: False


Processing with_cache:   0%|          | 1/266 [00:02<12:04,  2.74s/repo, repo=hn-notifier, stars=10, commits=1, total_prs=4, total_issues=0]Request GET /repos/mruby/mruby/issues/5954 failed with 403: Forbidden
Setting next backoff to 1327.983959s
Request GET /repos/iotexproject/iotex-desktop-wallet/issues/1664 failed with 403: Forbidden
Setting next backoff to 1327.968168s
Request GET /repos/dotintent/react-native-ble-plx/issues/1104 failed with 403: Forbidden
Setting next backoff to 1327.959039s
Request GET /repos/boutproject/BOUT-dev/issues/2531 failed with 403: Forbidden
Setting next backoff to 1327.778607s
Request GET /repos/OpenMage/magento-lts/issues/3975 failed with 403: Forbidden
Setting next backoff to 1327.713465s
Request GET /repos/coding-blocks/CBOnlineApp/issues/438 failed with 403: Forbidden
Setting next backoff to 1327.62854s
Request GET /repos/ohcnetwork/care/issues/2142 failed with 403: Forbidden
Setting next backoff to 1327.617344s
Request GET /repos/leo-editor/leo-ed